In [1]:
!pip install -q transformers datasets torchaudio librosa scikit-learn accelerate torchcodec

In [2]:
import os
import random
import torch
import torchaudio
import numpy as np


from torch.utils.data import Dataset
from transformers import (
Wav2Vec2Processor,
Wav2Vec2ForSequenceClassification,
TrainingArguments,
DataCollatorWithPadding,
Trainer
)


from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [3]:
# -------------------------
# Configuración general
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_PATH = "/content/dataset"
SAMPLE_RATE = 22050
NUM_LABELS = 3
SEED = 42


random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
# -------------------------
# Mapeo de emociones (3 clases)
# -------------------------
emotion2id = {
    "alegria": 0,      # Positivo
    "neutral": 1,      # Neutral
    "tristeza": 2,     # Negativo
    "miedo": 2,
    "sorpresa": 2,
    "enojo": 2,
    "asco": 2
}

id2emotion = {
    0: "positivo",
    1: "neutral",
    2: "negativo"
}


In [5]:
# -------------------------
# Dataset
# -------------------------
class EmotionDataset(Dataset):
    def __init__(self, files, data_dir, processor):
        self.files = files
        self.data_dir = data_dir
        self.processor = processor

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filename = self.files[idx]
        path = os.path.join(self.data_dir, filename)

        # Etiqueta desde el nombre
        emotion = filename.replace(".WAV","").replace(".wav","").split("_")[-1].lower()

        label = emotion2id[emotion]

        # Cargar audio
        speech, sr = torchaudio.load(path)

        if sr != SAMPLE_RATE:
            speech = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(speech)

        speech = speech.mean(dim=0)  # mono

        inputs = self.processor(
            speech.numpy(),
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=True
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }

In [6]:
# -------------------------
# Modelo y procesador
# -------------------------
MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-spanish"

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

data_collator = DataCollatorWithPadding(
    tokenizer=processor,
    padding=True,
    return_tensors="pt"
)

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to(DEVICE)

# Congelar extractor (clave para corpus pequeño)
for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-spanish
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 
projector.bias    | MISSING    | 
projector.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
# -------------------------
# División del corpus
# -------------------------
all_files = [f for f in os.listdir(DATASET_PATH) if f.lower().endswith(".wav")]
random.shuffle(all_files)

print("Archivos detectados:", len(all_files))
print(all_files[:5])


train_size = int(0.8 * len(all_files))
train_files = all_files[:train_size]
test_files = all_files[train_size:]

print(f"Train: {len(train_files)} | Test: {len(test_files)}")

train_dataset = EmotionDataset(train_files, DATASET_PATH, processor)
test_dataset = EmotionDataset(test_files, DATASET_PATH, processor)


Archivos detectados: 72
['026_001_MIEDO.wav', '037_001_NEUTRAL.wav', '032_001_TRISTEZA.wav', '056_001_MIEDO.wav', '052_001_ALEGRIA.wav']
Train: 57 | Test: 15


In [8]:
# -------------------------
# Métricas
# -------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1-score": f1
    }

In [9]:
# -------------------------
# Argumentos de entrenamiento
# -------------------------
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=8,
    warmup_steps=10,
    logging_dir="./logs",
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1-score",
    fp16=False
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
# -------------------------
# Trainer
# -------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
# -------------------------
# Entrenamiento y evaluación
# -------------------------
trainer.train()

results = trainer.evaluate()
print("\nResultados finales:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


ValueError: The model corresponding to this feature extractor: Wav2Vec2FeatureExtractor {
  "do_normalize": true,
  "feature_extractor_type": "Wav2Vec2FeatureExtractor",
  "feature_size": 1,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": true,
  "sampling_rate": 16000
}
 was trained using a sampling rate of 16000. Please make sure that the provided `raw_speech` input was sampled with 16000 and not 22050.